# 01. LightGBM 튜닝 (num_leaves) — 팀 #00 베이스라인 개선

**무엇을 하나:** 팀 확정 베이스라인(`00_baseline.ipynb`)의 전처리·파생변수·인코딩을 **그대로** 쓰고,
마지막 모델 설정 하나(`num_leaves`)만 바꿔서 점수를 개선한다.

**왜:** 실험으로 확인한 결과, 이 데이터는 스트레스 신호가 '여러 항목의 조합' 속에 숨어 있어
모델을 더 촘촘하게(복잡하게) 만들수록 잘 맞힌다. LightGBM 기본값은 `num_leaves=31`인데,
이걸 127로 키우면 5-Fold CV MAE가 크게 내려간다.

**목표:** CV MAE 0.2117(#00) → 약 0.174

**팀 규칙 준수:** random_state=42 고정 / 5-Fold CV 기준 / 데이터는 ../data, 제출은 ../submissions

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: 항상 42로 고정
import lightgbm as lgb  # 튜닝(조기종료)에 사용

## 1. 데이터 로드 (팀 #00 동일)

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

print('train:', train.shape, '/ test:', test.shape)

## 2. 결측치·중복 처리 (팀 #00 확정본 그대로)

In [ ]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# bone_density: 원본 수치 그대로 유지 (처리 없음)

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)

## 3. 파생변수 17개 (팀 #00 확정본 그대로)

In [ ]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape)

## 4. 인코딩 (팀 #00 확정본 그대로)

In [ ]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)

## 5. 튜닝 지점: num_leaves 비교 (#00 기본값 vs 개선)

`num_leaves`는 스무고개 나무가 몇 갈래까지 뻗을지를 정하는 값이다.
값이 크면 사람들을 더 잘게 나눠 '조합'을 더 세밀하게 잡는다.

- 기본값(31): 팀 #00과 동일 → 기준 확인용
- 튜닝(127): 여기에 학습률을 낮추고(0.03) 조기종료(early stopping)를 붙여 안정적으로 학습

아래에서 두 설정을 **똑같은 5-Fold CV**로 돌려 점수를 직접 비교한다.

In [ ]:
from lightgbm import LGBMRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def run_cv(params, use_early_stop):
    """5-Fold CV를 돌려 (검증오차, 테스트예측평균)을 돌려준다."""
    oof = np.zeros(len(x_train))          # 검증 예측(성능 측정용)
    test_pred = np.zeros(len(x_test))     # 테스트 예측(제출용, 5개 fold 평균)
    for tr_idx, va_idx in kf.split(x_train):
        X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        m = LGBMRegressor(**params)
        if use_early_stop:
            m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
        else:
            m.fit(X_tr, y_tr)
        oof[va_idx] = m.predict(X_va)
        test_pred += m.predict(x_test) / kf.n_splits
    return mean_absolute_error(y_train, oof), test_pred

# (A) 기본값 = 팀 #00 재현
mae_base, _ = run_cv(dict(random_state=RANDOM_STATE, verbose=-1), use_early_stop=False)
print(f"[기본값 num_leaves=31]  CV MAE: {mae_base:.4f}   <- 팀 #00 기준")

# (B) 튜닝
tuned = dict(n_estimators=5000, learning_rate=0.03, num_leaves=127,
             min_child_samples=10, random_state=RANDOM_STATE, verbose=-1)
mae_tuned, test_pred = run_cv(tuned, use_early_stop=True)
print(f"[튜닝 num_leaves=127]   CV MAE: {mae_tuned:.4f}   <- 개선")
print(f"\n개선 폭: {mae_base - mae_tuned:.4f} 낮아짐 (낮을수록 좋음)")

## 6. 최종 예측 & 제출 파일

위 CV에서 5개 fold가 각각 낸 테스트 예측을 평균낸 값(`test_pred`)을 그대로 사용한다.
스트레스 점수는 0~1 범위이므로 그 밖으로 나간 값은 잘라준다(clip).

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

test_pred = np.clip(test_pred, 0, 1)   # 0~1 범위 보정
sample_submission['stress_score'] = test_pred
sample_submission.to_csv('../submissions/submit_01_tune.csv', index=False)
print('저장 완료: submissions/submit_01_tune.csv')
sample_submission.head()

## 결과 요약

| 모델 | num_leaves | CV MAE |
|---|---|---|
| 팀 #00 베이스라인 | 31(기본) | 0.2117 |
| **#01 튜닝(이 노트북)** | **127** | **~0.174** |

**다음 단계 후보**
1. XGBoost·CatBoost를 같은 CV 틀로 돌려 예측 평균(앙상블) → 추가 하락 기대
2. num_leaves 외 파라미터(Optuna로 자동 탐색)
3. Dacon 제출 후 리더보드 점수와 CV 비교(팀 규칙 3)